# The same data every time: seeds for tests and CI

Tests need data that does not change between runs. A saved schema plus a seed gives byte-for-byte the same rows whenever you ask, so a failing test fails for the same reason tomorrow.

**Uses:** 8,000 Core rows (four runs of 2,000), no AI rows. API reference: [generaterealdata.com/docs/api/](https://generaterealdata.com/docs/api/).

## Setup

You need an API key (PRO, ULTRA or Enterprise plan): sign in at [generaterealdata.com](https://generaterealdata.com), open the dashboard and click **Generate API key**. Keep it out of the notebook: put it in the `GRD_API_KEY` environment variable, or in Colab's **Secrets** panel under the same name.

In [1]:
import os
import time
from pathlib import Path

import requests

API = "https://generaterealdata.com/api/v1"
KEY = os.environ.get("GRD_API_KEY")
if not KEY:                                   # Google Colab: read it from the Secrets panel
    from google.colab import userdata
    KEY = userdata.get("GRD_API_KEY")
HEADERS = {"Authorization": f"Bearer {KEY}"}
OUT = Path("downloads")
OUT.mkdir(exist_ok=True)


def api(method, path, **kwargs):
    """One API call. Waits and tries again if the API says to slow down or another dataset is still running."""
    while True:
        r = requests.request(method, API + path, headers=HEADERS, timeout=90, **kwargs)
        busy = r.status_code == 409 and r.json().get("error", {}).get("code") == "run_in_progress"
        if r.status_code == 429 or busy:
            time.sleep(int(r.headers.get("Retry-After", 15)))
            continue
        data = r.json()
        if r.status_code >= 400:
            e = data["error"]
            raise RuntimeError(f"{e['code']}: {e['message']} {e.get('hint', '')}")
        return data


def generate(spec):
    """Start a dataset and wait until it is ready. Returns its final status."""
    ds = api("POST", "/datasets", json=spec)
    while ds["status"] not in ("ready", "failed"):
        time.sleep(5)
        ds = api("GET", f"/datasets/{ds['id']}")
    if ds["status"] == "failed":
        raise RuntimeError(ds["error"]["message"])
    return ds


def download(ds, filename, file=None):
    """Save a ready dataset's file. file="db" fetches the SQLite file of an SQL dataset."""
    path = OUT / filename
    url = f"{API}/datasets/{ds['id']}/download"
    with requests.get(url, params={"file": file} if file else None, headers=HEADERS, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    return path


def summary(ds):
    """The facts worth reading from a dataset's status."""
    keys = ("status", "format", "rows_generated", "rows_per_table", "credits_charged", "ai_rows_charged", "schema_id", "seed")
    return {k: ds[k] for k in keys if k in ds}

## 1. Design once, keep the `schema_id`

In [2]:
def repo_file(path):
    """Read a file of this repository: from disk when the repo is cloned, from GitHub otherwise (e.g. in Colab)."""
    local = Path("..") / path
    if local.exists():
        return local.read_text(encoding="utf-8")
    r = requests.get("https://raw.githubusercontent.com/generaterealdata/api-examples/main/" + path, timeout=30)
    r.raise_for_status()
    return r.text


SHOP_SQL = repo_file("schemas/shop.sql")
SHOP_RULES = repo_file("schemas/shop_rules.txt").strip()

first = generate({"mode": "sql", "sql": SHOP_SQL, "notes": SHOP_RULES, "rows": 2000, "format": "csv", "seed": 5})
SCHEMA_ID = first["schema_id"]
summary(first)

{'status': 'ready',
 'format': 'csv',
 'rows_generated': 2000,
 'rows_per_table': {'order_items': 1150,
  'orders': 600,
  'products': 50,
  'customers': 200},
 'credits_charged': 2000,
 'ai_rows_charged': 0,
 'schema_id': 'sch_J-RM62xFhnvKTpFA',
 'seed': 5}

## 2. Same schema, same seed: the same data

Two separate runs with seed 42. `new_run` is needed here only because an identical request within 10 minutes is otherwise treated as an accidental duplicate and answered with the first dataset.

In [3]:
import hashlib
import zipfile


def fingerprints(ds, filename):
    """SHA-256 of each table's CSV."""
    archive = zipfile.ZipFile(download(ds, filename))
    return {name: hashlib.sha256(archive.read(name)).hexdigest()[:16] for name in sorted(archive.namelist())}


spec = {"mode": "schema", "schema_id": SCHEMA_ID, "rows": 2000, "format": "csv", "new_run": True}
run_a = fingerprints(generate({**spec, "seed": 42}), "seed42_a.zip")
run_b = fingerprints(generate({**spec, "seed": 42}), "seed42_b.zip")
run_a

{'customers.csv': '837f2f1f49e3736d',
 'order_items.csv': 'cd7fe32bc5dbcbd5',
 'orders.csv': '5a25695acd666521',
 'products.csv': '8f99fca321f865c7'}

In [4]:
print("Run A and run B identical:", run_a == run_b)

Run A and run B identical: True


## 3. A different seed: different data, same structure

In [5]:
run_c = fingerprints(generate({**spec, "seed": 43}), "seed43.zip")
print("Seed 43 differs from seed 42:", run_c != run_a)
print("Same files:", list(run_c) == list(run_a))

Seed 43 differs from seed 42: True
Same files: True


## 4. Use it in tests

Keep the `schema_id` and the seed in your repository and fetch the data when the test session starts. A `pytest` fixture:

```python
# conftest.py
import os, sqlite3, time, requests, pytest

API = "https://generaterealdata.com/api/v1"
HEADERS = {"Authorization": f"Bearer {os.environ['GRD_API_KEY']}"}


@pytest.fixture(scope="session")
def shop_db(tmp_path_factory):
    spec = {"mode": "schema", "schema_id": "sch_...", "rows": 2000, "format": "sql", "seed": 42}
    ds = requests.post(f"{API}/datasets", json=spec, headers=HEADERS, timeout=70).json()
    while ds["status"] not in ("ready", "failed"):
        time.sleep(5)
        ds = requests.get(f"{API}/datasets/{ds['id']}", headers=HEADERS, timeout=30).json()
    path = tmp_path_factory.mktemp("data") / "shop.sqlite"
    path.write_bytes(requests.get(f"{API}/datasets/{ds['id']}/download?file=db", headers=HEADERS, timeout=300).content)
    return sqlite3.connect(path)
```

In CI, store the key as a secret (`GRD_API_KEY`) and cache the downloaded file keyed on the schema id and seed, so most pipeline runs do not call the API at all.

## Good to know

- Without a `seed` the service picks one and returns it in the status, so any dataset can be repeated later.
- The guarantee is for the same `schema_id` (or the same inline schema) and the same `rows`. Designing again from a description or DDL can produce a different schema.
- Saved schemas stay until you delete them (`DELETE /schemas/{id}`).